# 02_data_cleaning
This notebook carries the cleaned drillhole and assay tables from Notebook 01 through a geology-aware structural review. It is written for a non-geologist audience, with explanations that make the data workflow understandable to employers outside the mining and exploration industry.


## Objectives
- Load the cleaned drill and assay datasets produced in Notebook 01.
- Restore the correct drillhole structure after Excel merged cells caused missing IDs.
- Convert depth and sample fields into numeric values for validation.
- Separate geological interval data into lithology, alteration, and mineralization tables.
- Validate the interval structure and standardize descriptive geology fields.
- Export clean tables for later geochemical merging and analysis.


## 1. Setup and libraries
We start with the Python packages needed for importing data, checking structure, and preparing the cleaned files.


In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

project_root = Path().resolve().parent
raw_data = project_root / "Data" / "raw"
clean_data = project_root / "Data" / "clean"

raw_data, clean_data


(PosixPath('/home/panchovilla/Code/Projects/Portfolio/Exploration Analytics/Data/raw'),
 PosixPath('/home/panchovilla/Code/Projects/Portfolio/Exploration Analytics/Data/clean'))

## 2. Load the cleaned raw datasets
Notebook 01 created `drill_raw.csv` and `assay_raw.csv` after importing the original source files. These are our starting point for interval cleaning.


In [11]:
drill_df = pd.read_csv(raw_data / "drill_raw.csv")
assay_df = pd.read_csv(raw_data / "assay_raw.csv")

print(f"drill_df: {drill_df.shape[0]} rows x {drill_df.shape[1]} columns")
print(f"assay_df: {assay_df.shape[0]} rows x {assay_df.shape[1]} columns")


drill_df: 1430 rows x 21 columns
assay_df: 6635 rows x 116 columns


### What geology interval data means
Drillcore data is recorded as intervals down a hole. Each row describes a piece of the hole between a `from` depth and a `to` depth.
- **Lithology** describes rock type, texture, and appearance.
- **Alteration** describes chemical changes in the rock caused by fluids.
- **Mineralization** records where economically important minerals appear.
For a non-geologist audience: this notebook is making sure these separate types of observations are clean and easy to work with, so later analysis is reliable and the story is clear.


## 3. Restore hole identifiers
In the original Excel drill logs, geologists often wrote the hole name once and then logged many rows beneath it. When these merged cells were imported, the rows below lost the hole ID and became `NaN`.
For a general audience, this means the table looked like it was missing its “address labels”, even though the geological observations were still there. We fix that by copying the last known hole ID down the table.


In [12]:
missing_before = drill_df[drill_df["hole_number"].isna()]
print(f"Rows missing hole_number before fill: {len(missing_before)}")
missing_before.head(5)


Rows missing hole_number before fill: 526


,hole_number,from,to,base_lithology,litho_detailed,lith_general,lith_grain_size,lith_texture1,lith_texture2,lith_texture3,...,alt_from_m,alt_to_m,dominant_alteration,intensity,dominant_alt_mins,min_from_m,min_to_m,py_pct_nv,ccp_pct_nv,bn_pct_nv
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.20,17.00,PHY,4.0,Ser+py,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,26.32,38.00,SKN,7.0,Magnetite+garnet+ep,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,35.0,44.00,0.1,0.0,0.0
18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,152.55,154.02,SKN,5.0,Magnetite+ep+garnet,143.7,154.02,2,0.2,0.0
22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,181.50,200.00,SKN,5.0,Magnetite+ep+garnet,170.0,185.25,0.1,0.0,0.0


In [13]:
drill_df["hole_number"] = drill_df["hole_number"].ffill()
missing_after = drill_df[drill_df["hole_number"].isna()]
print(f"Rows missing hole_number after fill: {len(missing_after)}")


Rows missing hole_number after fill: 0


## 4. Convert interval and sample values to numeric
Geological depth values and assay amounts must behave like numbers, not text. This allows us to check for bad intervals, compare depths, and merge the geology with assay data later.


In [14]:
numeric_columns = [
    "from", "to",
    "alt_from_m", "alt_to_m",
    "min_from_m", "min_to_m",
    "py_pct_nv", "ccp_pct_nv", "bn_pct_nv"
]

for col in numeric_columns:
    if col in drill_df.columns:
        drill_df[col] = pd.to_numeric(drill_df[col], errors="coerce")

for col in ["from", "to"]:
    if col in assay_df.columns:
        assay_df[col] = pd.to_numeric(assay_df[col], errors="coerce")

drill_df[[c for c in numeric_columns if c in drill_df.columns]].dtypes


from          float64
to            float64
alt_from_m    float64
alt_to_m      float64
min_from_m    float64
min_to_m      float64
py_pct_nv     float64
ccp_pct_nv    float64
bn_pct_nv     float64
dtype: object

## 5. Identify each interval type
The drill log mixes three different geological interval systems in one table. Here we separate them by the columns that are filled: lithology rows, alteration rows, and mineralization rows.
This is important because each interval system has a different purpose and should be analyzed separately.


In [15]:
def get_interval_type(row):
    lithology_filled = not pd.isna(row.get("from")) and not pd.isna(row.get("to"))
    alteration_filled = not pd.isna(row.get("alt_from_m")) and not pd.isna(row.get("alt_to_m"))
    mineralization_filled = not pd.isna(row.get("min_from_m")) and not pd.isna(row.get("min_to_m"))

    if lithology_filled and not alteration_filled and not mineralization_filled:
        return "Lithology"
    if alteration_filled and not lithology_filled and not mineralization_filled:
        return "Alteration"
    if mineralization_filled and not lithology_filled and not alteration_filled:
        return "Mineralization"
    return "Mixed/Unknown"

drill_df["interval_type"] = drill_df.apply(get_interval_type, axis=1)
print(drill_df["interval_type"].value_counts(dropna=False))


interval_type
Mixed/Unknown     1016
Alteration         207
Mineralization     130
Lithology           77
Name: count, dtype: int64


In [16]:
lith_df = drill_df[drill_df["interval_type"] == "Lithology"].copy()
alt_df = drill_df[drill_df["interval_type"] == "Alteration"].copy()
min_df = drill_df[drill_df["interval_type"] == "Mineralization"].copy()

print(f"Lithology rows: {len(lith_df)}")
print(f"Alteration rows: {len(alt_df)}")
print(f"Mineralization rows: {len(min_df)}")


Lithology rows: 77
Alteration rows: 207
Mineralization rows: 130


## 6. Validate interval quality
Here we check whether depth intervals make sense. We look for intervals where the top depth is below the bottom depth, intervals of zero length, and overlapping intervals in the same hole.
These checks are like quality control for the geological record. If a drill interval is reversed or overlaps, it means the table needs to be corrected before analysis.


In [17]:
def validate_intervals(df, name, from_col="from", to_col="to"):
    print(f"--- {name} validation ---")
    if from_col not in df.columns or to_col not in df.columns:
        print(f"Missing columns {from_col} or {to_col} in {name}")
        return
    reversed_intervals = df[df[from_col] > df[to_col]]
    zero_length = df[df[from_col] == df[to_col]]
    print(f"Reversed intervals: {len(reversed_intervals)}")
    print(f"Zero-length intervals: {len(zero_length)}")

    overlap_count = 0
    for hole, group in df.sort_values(["hole_number", from_col]).groupby("hole_number"):
        prev_to = group[to_col].shift(1)
        overlaps = group[group[from_col] < prev_to]
        overlap_count += len(overlaps)
    print(f"Overlapping intervals: {overlap_count}")

validate_intervals(lith_df, "Lithology", from_col="from", to_col="to")
validate_intervals(alt_df, "Alteration", from_col="alt_from_m", to_col="alt_to_m")
validate_intervals(min_df, "Mineralization", from_col="min_from_m", to_col="min_to_m")


--- Lithology validation ---
Reversed intervals: 0
Zero-length intervals: 0
Overlapping intervals: 0
--- Alteration validation ---
Reversed intervals: 0
Zero-length intervals: 0
Overlapping intervals: 0
--- Mineralization validation ---
Reversed intervals: 0
Zero-length intervals: 0
Overlapping intervals: 0


## 7. Standardize descriptive geology fields
Geological descriptions are often entered in different ways by different loggers. To make this data easier to work with, we standardize text fields by removing extra spaces and using consistent capitalization.
This is similar to cleaning product categories in a business dataset so that everyone can compare the same things.


In [18]:
def clean_categories(df, columns):
    for col in columns:
        if col in df.columns:
            cleaned = df[col].astype(str).str.strip()
            cleaned = cleaned.where(~df[col].isna())
            cleaned = cleaned.str.replace(r"\s+", " ", regex=True).str.upper()
            df[col] = cleaned
    return df

lith_df = clean_categories(lith_df, ["base_lithology", "litho_detailed", "lith_description"])
alt_df = clean_categories(alt_df, ["dominant_alteration", "intensity", "dominant_alt_mins"])
min_df = clean_categories(min_df, ["dominant_alt_mins"])

print("Example cleaned lithology descriptions:")
display(lith_df[["hole_number", "from", "to", "base_lithology", "lith_description"]].head(5))


Example cleaned lithology descriptions:


,hole_number,from,to,base_lithology,lith_description
0,AXE-23-001,0.0,2.0,OVB,OVERBURDEN
1,AXE-23-001,2.0,6.0,CL,"TOP OF BEDROCK REACHED AT 2M, NO RECOVERY UNTI..."
10,AXE-23-001,44.0,55.0,CL,"CORE LOSS IN FAULTY MATERIAL, DRILLER HAD TO R..."
76,AXE-23-002,0.0,2.0,OVB,"OVERBURDEN, NO CORE RECOVERED"
77,AXE-23-002,2.0,6.0,CL,"BEDROCK HIT AT 2M, NO CORE RECOVERED"


## 8. Handle holes without assay records
Some drillholes were logged geologically but never sent for chemical assay. These holes are still useful for geological context, but they cannot support geochemical analysis.
We keep them in the geology tables, but exclude them from the next geochemistry-focused merge step.


In [19]:
holes_with_no_assays = sorted(set(drill_df["hole_number"]) - set(assay_df["hole_number"]))
print(f"Drillholes with no assay records: {len(holes_with_no_assays)}")
holes_with_no_assays[:10]


Drillholes with no assay records: 3


['AXE-23-020', 'MPD-23-004', 'MPD-23-011']

In [20]:
lith_df = lith_df[~lith_df["hole_number"].isin(holes_with_no_assays)].copy()
alt_df = alt_df[~alt_df["hole_number"].isin(holes_with_no_assays)].copy()
min_df = min_df[~min_df["hole_number"].isin(holes_with_no_assays)].copy()

print(f"Lithology holes after assay filter: {lith_df["hole_number"].nunique()}")
print(f"Alteration holes after assay filter: {alt_df["hole_number"].nunique()}")
print(f"Mineralization holes after assay filter: {min_df["hole_number"].nunique()}")


Lithology holes after assay filter: 30
Alteration holes after assay filter: 26
Mineralization holes after assay filter: 21


## 9. Export cleaned tables for Notebook 03
These cleaned interval tables are now ready for geochemical merging, visualization, and reporting. The export files preserve the geology structure and make the next stage of the workflow transparent.


In [21]:
clean_data.mkdir(parents=True, exist_ok=True)
lith_df.to_csv(clean_data / "lithology_clean.csv", index=False)
alt_df.to_csv(clean_data / "alteration_clean.csv", index=False)
min_df.to_csv(clean_data / "mineralization_clean.csv", index=False)
assay_df.to_csv(clean_data / "assays_clean.csv", index=False)
print("Exported cleaned interval and assay tables to Data/clean.")


Exported cleaned interval and assay tables to Data/clean.


## Summary
- The drillhole ID values were restored so every geological interval is attached to the correct hole.
- Numeric depth and assay fields were standardized for validation and future merges.
- Lithology, alteration, and mineralization intervals were separated into distinct tables.
- Interval structure was validated for reversed, zero-length, and overlapping records.
- Categorical geology descriptions were standardized to make the dataset easier to interpret.
- Clean tables were exported for Notebook 03, where geochemical merging and analysis will begin.
